In [8]:
import pandas as pd
import numpy as np
import plotly.express as px

In [9]:
data=pd.read_csv(r"csvs\Topic_Distribution_Unequal.csv")

In [10]:
data

,Year,Title,Extracted_Text,Region/Authority,Constituency,Processed_Text,Bigrams,Trigrams,Topic_Counts,Total_Topic_Count,Climate Change,Economic Growth,Debt,Crisis,Risk,Reform
0,2024,"IMFC Statement by Christine Lagarde, President...",\r\n \r\n INTERNATIONAL MONETARY AND FINANCIA...,European Central Bank,OBS,"['international', 'financial', 'committee', 'f...","[('international', 'financial'), ('financial',...","[('international', 'financial', 'committee'), ...","{'Climate Change': 2, 'Economic Growth': 12, '...",167,0.011976,0.071856,0.005988,0.395210,0.514970,0.000000
1,2024,"IMFC Statement by HE Haitham Al Ghais, Secreta...",\r\n \r\n INTERNATIONAL MONETARY AND FINANCIA...,Organization of the Petroleum Exporting Countries,OBS,"['international', 'financial', 'committee', 'f...","[('international', 'financial'), ('financial',...","[('international', 'financial', 'committee'), ...","{'Climate Change': 0, 'Economic Growth': 30, '...",105,0.000000,0.285714,0.000000,0.247619,0.466667,0.000000
2,2024,"IMFC Statement by Ayman Al-Sayari, Governor of...",\r\n \r\n INTERNATIONAL MONETARY AND FINANCIA...,Saudi Arabia,SA,"['international', 'financial', 'committee', 'f...","[('international', 'financial'), ('financial',...","[('international', 'financial', 'committee'), ...","{'Climate Change': 0, 'Economic Growth': 12, '...",227,0.000000,0.052863,0.057269,0.528634,0.352423,0.008811
3,2024,"IMFC Statement by Antoine Armand, Minister of ...",INTERNATIONAL MONETARY AND FINANCIAL COMMITTE...,France,FF,"['international', 'financial', 'committee', 'f...","[('international', 'financial'), ('financial',...","[('international', 'financial', 'committee'), ...","{'Climate Change': 2, 'Economic Growth': 1, 'D...",103,0.019417,0.009709,0.038835,0.582524,0.330097,0.019417
4,2024,"IMFC Statement by Luis Caputo, Minister of Eco...",\r\n \r\n INTERNATIONAL MONETARY AND FINANCIA...,Argentina,AG,"['international', 'financial', 'committee', 'f...","[('international', 'financial'), ('financial',...","[('international', 'financial', 'committee'), ...","{'Climate Change': 3, 'Economic Growth': 31, '...",444,0.006757,0.069820,0.063063,0.454955,0.391892,0.013514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
559,2004,IMFC Statement by the Honorable Domenico Sinis...,\r\n International Monetary and \r\nFinancial...,Italy,IT,"['international', 'financial', 'committee', 't...","[('international', 'financial'), ('financial',...","[('international', 'financial', 'committee'), ...","{'Climate Change': 0, 'Economic Growth': 29, '...",237,0.000000,0.122363,0.075949,0.426160,0.350211,0.025316
560,2004,"IMFC Statement by the Honorable John W. Snow, ...",\r\n International Monetary and \r\nFinancial...,United States,US,"['international', 'financial', 'committee', 't...","[('international', 'financial'), ('financial',...","[('international', 'financial', 'committee'), ...","{'Climate Change': 0, 'Economic Growth': 12, '...",109,0.000000,0.110092,0.091743,0.385321,0.376147,0.036697
561,2004,IMFC Statement by H.E. Sadakazu Tanigaki Minis...,\r\n International Monetary and \r\nFinancial...,Japan,JA,"['international', 'financial', 'committee', 't...","[('international', 'financial'), ('financial',...","[('international', 'financial', 'committee'), ...","{'Climate Change': 0, 'Economic Growth': 4, 'D...",127,0.000000,0.031496,0.047244,0.535433,0.338583,0.047244
562,2004,"IMFC Statement By James D. Wolfensohn, Preside...",\r\n International Monetary and \r\nFinancial...,World Bank,OBS,"['international', 'financial', 'committee', 't...","[('international', 'financial'), ('financial',...","[('international', 'financial', 'committee'), ...","{'Climate Change': 0, 'Economic Growth': 18, '...",152,0.000000,0.118421,0.046053,0.493421,0.315789,0.026316


In [11]:
score_columns = ["Climate Change", "Economic Growth", "Debt ", "Crisis", "Risk", "Reform"]
data_melted = data.melt(id_vars=["Year", "Region/Authority"], value_vars=score_columns, 
                         var_name="Score Type", value_name="Score")

# Ensure Score column is numeric and drop NaN values
data_melted["Score"] = pd.to_numeric(data_melted["Score"], errors="coerce")
data_melted = data_melted.dropna(subset=["Score"])

# Aggregate scores by Year and Score Type (calculate average)
data_grouped = data_melted.groupby(["Year", "Score Type"], as_index=False)["Score"].mean()

# Plotly line chart
fig = px.line(data_grouped, x="Year", y="Score", color="Score Type", 
              line_group="Score Type", markers=True,
              title="Yearly Scores by Category (Averaged)",
              labels={"Score": "Average Score", "Year": "Year"})

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Average Score",
    legend_title="Score Type",
    hovermode="x unified"
)

fig.show()

In [12]:
# Aggregate scores by Score Type (calculate average over all documents)
data_avg = data_melted.groupby("Score Type", as_index=False)["Score"].mean()

# Sort scores from highest to lowest
data_avg = data_avg.sort_values(by="Score", ascending=False)

# Create bar chart
fig = px.bar(data_avg, x="Score", y="Score Type", orientation="h", 
             title="Average Scores Across All Documents",
             labels={"Score": "Average Score", "Score Type": "Score Category"},
             text="Score")

fig.update_layout(
    xaxis_title="Average Score",
    yaxis_title="Score Category",
    yaxis=dict(categoryorder="total ascending"),  # Ensure highest score is at the top
    showlegend=False
)

fig.show()

In [13]:
# Calculate standard deviation
data_std = data_melted.groupby(["Year", "Score Type"], as_index=False)["Score"].std()
data_std.rename(columns={"Score": "Score Std Dev"}, inplace=True)

# Merge average and standard deviation data
data_combined = pd.merge(data_grouped, data_std, on=["Year", "Score Type"])

# Plotly line chart for standard deviation
fig_std = px.line(data_combined, x="Year", y="Score Std Dev", color="Score Type", 
                  line_group="Score Type", markers=True,
                  title="Yearly Score Standard Deviation by Category",
                  labels={"Score Std Dev": "Standard Deviation of Score", "Year": "Year"})

fig_std.update_layout(
    xaxis_title="Year",
    yaxis_title="Standard Deviation of Score",
    legend_title="Score Type",
    hovermode="x unified"
)

fig_std.show()

In [14]:
data['Region/Authority'].value_counts()

Region/Authority
Germany                                           21
Japan                                             21
Switzerland                                       21
Italy                                             20
European Commission                               20
                                                  ..
Peru                                               1
Burkina Faso                                       1
Colombia                                           1
Lithuania                                          1
International Monetary and Financial Committee     1
Name: count, Length: 68, dtype: int64

In [15]:
import plotly.graph_objects as go

def plot_region_average_scores(data, selected_region):
    """
    Create an interactive bar plot using Plotly showing average scores 
    for all score columns for a selected region.
    
    Parameters:
    -----------
    data : pandas.DataFrame
        DataFrame containing Year, Region/Authority and score columns
    selected_region : str
        The region to filter and analyze
    
    Returns:
    --------
    plotly.graph_objects.Figure
        Interactive Plotly figure that can be displayed or saved
    """
    # Filter data for the selected region
    region_data = data[data['Region/Authority'] == selected_region]
    
    # Define score columns (all columns except Year and Region/Authority)
    score_columns = ['Climate Change', 'Economic Growth', 'Debt ', 'Crisis', 'Risk', 'Reform']
    
    # Calculate average scores for each category
    avg_scores = region_data[score_columns].mean()
    
    # Create a DataFrame for plotting
    plot_data = pd.DataFrame({
        'Category': avg_scores.index,
        'Average Score': avg_scores.values
    })
    
    # Create interactive bar plot with Plotly
    fig = px.bar(
        plot_data, 
        x='Category', 
        y='Average Score',
        title=f'Average Scores for {selected_region}',
        text_auto='.2f',  # Display values on bars with 2 decimal places
        color='Average Score',
        color_continuous_scale='viridis',
        height=600
    )
    
    # Customize layout
    fig.update_layout(
        xaxis_title='Category',
        yaxis_title='Average Score',
        title={
            'text': f'Average Scores for {selected_region}',
            'y':0.95,
            'x':0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': {'size': 20}
        },
        xaxis={'categoryorder':'total descending'},  # Sort bars by value
        plot_bgcolor='rgba(240,240,240,0.8)'  # Light gray background
    )
    
    # Add more customization to text on bars
    fig.update_traces(
        textfont_size=14,
        textangle=0,
        textposition="outside",
        cliponaxis=False
    )
    
    return fig

In [16]:
fig = plot_region_average_scores(data, 'Germany')
fig.show()